<a href="https://colab.research.google.com/github/hadri1-bisch/SCM/blob/main/scm_survey_part_3_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

cModélisation du questionnaire:

In [ ]:
import itertools
import pandas as pd
import numpy as np


In [ ]:
# Définition des niveaux d'attributs
prix = [100, 150, 300]
hygiene = [1, 2]     # 1: seconde main, 2: standard
assortiment = [1, 2] # 1: Limité, 2: Extensif
garantie = [1, 2]    # 1: Sans retour, 2: Retours 30j

# Génération des 24 profils uniques
profils = list(itertools.product(prix, hygiene, assortiment, garantie))
df_profils = pd.DataFrame(profils, columns=['Prix', 'Hygiene', 'Assortiment', 'Garantie'])
df_24_profils = pd.DataFrame(profils, columns=['Prix', 'Hygiène', 'Assortiment', 'Garantie'])
print(df_24_profils.to_string())

    Prix  Hygiène  Assortiment  Garantie
0    100        1            1         1
1    100        1            1         2
2    100        1            2         1
3    100        1            2         2
4    100        2            1         1
5    100        2            1         2
6    100        2            2         1
7    100        2            2         2
8    150        1            1         1
9    150        1            1         2
10   150        1            2         1
11   150        1            2         2
12   150        2            1         1
13   150        2            1         2
14   150        2            2         1
15   150        2            2         2
16   300        1            1         1
17   300        1            1         2
18   300        1            2         1
19   300        1            2         2
20   300        2            1         1
21   300        2            1         2
22   300        2            2         1
23   300        

suppression des profils irréaliste il reste:
- Seconde main : 2 prix (100 et 150) × 1 niveau de disponibilité des tailles × 2 conditions de retour = 4 profils.
- Neuf : 3 prix (100, 150 et 300) × 2 niveaux de disponibilité des tailles × 2 conditions de retour = 12 profils.

In [ ]:

# Filtrage des profils selon les contraintes définies
def filtre_profils(row):
    statut = row['Hygiène']  # 1 = seconde main, 2 = neuf
    prix = row['Prix']
    assortiment = row['Assortiment']

    # Contraintes pour la seconde main
    if statut == 1:
        if prix == 300:
            return False  # Prix maximum exclu
        if assortiment == 2:
            return False  # Plusieurs tailles exclues

    return True

# Application du filtre
df_profils_valides = (
    df_24_profils[
        df_24_profils.apply(filtre_profils, axis=1)
    ]
    .reset_index(drop=True)
)

# Affichage des résultats
print(f"Nombre de profils valides : {len(df_profils_valides)}")
print(df_profils_valides.to_string(index=False))


Nombre de profils valides : 16
 Prix  Hygiène  Assortiment  Garantie
  100        1            1         1
  100        1            1         2
  100        2            1         1
  100        2            1         2
  100        2            2         1
  100        2            2         2
  150        1            1         1
  150        1            1         2
  150        2            1         1
  150        2            1         2
  150        2            2         1
  150        2            2         2
  300        2            1         1
  300        2            1         2
  300        2            2         1
  300        2            2         2


créations de toutes les paires candidates:

In [ ]:
from itertools import product

In [ ]:
# Séparer les profils selon leur statut
profils_seconde_main = df_profils_valides[
    df_profils_valides["Hygiène"] == 1
].copy()

profils_neufs = df_profils_valides[
    df_profils_valides["Hygiène"] == 2
].copy()

# Générer toutes les paires : seconde main vs neuf
paires = []

for (idx_sm, profil_sm), (idx_neuf, profil_neuf) in product(
    profils_seconde_main.iterrows(),
    profils_neufs.iterrows()
):
    paires.append({
        "Prix_SM": profil_sm["Prix"],
        "Tailles_SM": profil_sm["Assortiment"],
        "Retours_SM": profil_sm["Garantie"],
        "Prix_neuf": profil_neuf["Prix"],
        "Tailles_neuf": profil_neuf["Assortiment"],
        "Retours_neuf": profil_neuf["Garantie"]
    })

df_paires_candidates = pd.DataFrame(paires)

print(f"Nombre de paires candidates : {len(df_paires_candidates)}")
print(df_paires_candidates.to_string(index=False))


Nombre de paires candidates : 48
 Prix_SM  Tailles_SM  Retours_SM  Prix_neuf  Tailles_neuf  Retours_neuf
     100           1           1        100             1             1
     100           1           1        100             1             2
     100           1           1        100             2             1
     100           1           1        100             2             2
     100           1           1        150             1             1
     100           1           1        150             1             2
     100           1           1        150             2             1
     100           1           1        150             2             2
     100           1           1        300             1             1
     100           1           1        300             1             2
     100           1           1        300             2             1
     100           1           1        300             2             2
     100           1           

In [ ]:
pip install choicedesign

méthode aléatoire:

In [ ]:
import choicedesign as cd

In [ ]:

# Sélectionner 12 paires parmi les 48 candidates
df_questions = df_paires_candidates.sample(
    n=12,
    random_state=42
).reset_index(drop=True)

# Numéroter les questions
df_questions.insert(0, "Question", range(1, len(df_questions) + 1))

print(f"Nombre de questions sélectionnées : {len(df_questions)}")
display(df_questions)


Nombre de questions sélectionnées : 12


,Question,Prix_SM,Tailles_SM,Retours_SM,Prix_neuf,Tailles_neuf,Retours_neuf
0,1,150,1,1,100,2,2
1,2,150,1,2,150,1,1
2,3,150,1,1,100,2,1
3,4,150,1,2,150,2,2
4,5,150,1,1,100,1,1
5,6,150,1,2,100,1,2
6,7,100,1,2,100,1,1
7,8,100,1,2,150,2,2
8,9,100,1,1,150,1,1
9,10,150,1,1,100,1,2


méthode d-efficency (a revoir)

In [ ]:
from choicedesign.design import EffDesign
from choicedesign.expressions import Attribute, Parameter, ASC

# --------------------------------------------------
# 1. Définir les attributs des profils existants
# --------------------------------------------------

prix_sm = Attribute("Prix_SM", [100, 150])
retours_sm = Attribute("Retours_SM", [1, 2])

prix_neuf = Attribute("Prix_neuf", [100, 150, 300])
tailles_neuf = Attribute("Tailles_neuf", [1, 2])
retours_neuf = Attribute("Retours_neuf", [1, 2])

# --------------------------------------------------
# 2. Définir les paramètres du modèle
# Priors nuls : tous les coefficients commencent à 0
# --------------------------------------------------

beta_prix_150 = Parameter("beta_prix_150", 0.0)
beta_prix_300 = Parameter("beta_prix_300", 0.0)
beta_tailles = Parameter("beta_tailles", 0.0)
beta_retours = Parameter("beta_retours", 0.0)

# Constante spécifique à l'alternative seconde main
asc_sm = ASC("asc_sm", 0.0)

# --------------------------------------------------
# 3. Définir les fonctions d'utilité
# Le prix est catégoriel, avec 100 CHF comme référence
# --------------------------------------------------

V_sm = (
    asc_sm
    + beta_prix_150 * (prix_sm == 150)
    + beta_retours * (retours_sm == 2)
)

V_neuf = (
    beta_prix_150 * (prix_neuf == 150)
    + beta_prix_300 * (prix_neuf == 300)
    + beta_tailles * (tailles_neuf == 2)
    + beta_retours * (retours_neuf == 2)
)

V = {
    1: V_sm,
    2: V_neuf
}

# --------------------------------------------------
# 4. Préparer les 48 paires candidates
# --------------------------------------------------

colonnes_modele = [
    "Prix_SM",
    "Retours_SM",
    "Prix_neuf",
    "Tailles_neuf",
    "Retours_neuf"
]

candidates = df_paires_candidates.reset_index(drop=True).copy()

assert len(candidates) == 48, (
    f"Il faut 48 paires candidates, mais il y en a {len(candidates)}."
)

# --------------------------------------------------
# 5. Créer l'évaluateur D-efficient
# --------------------------------------------------

design = EffDesign(
    X=[
        prix_sm,
        retours_sm,
        prix_neuf,
        tailles_neuf,
        retours_neuf
    ],
    ncs=12
)


def evaluer_selection(indices):
    plan = candidates.iloc[indices][colonnes_modele].copy()

    # Identifier les 12 situations de choix
    plan.insert(0, "CS", range(1, len(plan) + 1))

    # Calculer le D-error et l'équilibre des utilités
    d_error, equilibre = design.evaluate(
        plan,
        V=V,
        criterion="d"
    )

    return float(d_error), float(equilibre)

# --------------------------------------------------
# 6. Recherche locale avec plusieurs départs aléatoires
# --------------------------------------------------

rng = np.random.default_rng(42)

meilleur_plan = None
meilleur_d_error = np.inf
meilleur_equilibre = None

nombre_depart = 5
iterations_max = 30

for depart in range(nombre_depart):

    selection = list(
        rng.choice(len(candidates), size=12, replace=False)
    )

    d_error, equilibre = evaluer_selection(selection)

    for iteration in range(iterations_max):

        indices_selectionnes = set(selection)
        meilleur_echange = None
        meilleur_d_local = d_error
        meilleur_eq_local = equilibre

        # Tester le remplacement d'une question par une autre
        for position in range(12):

            for candidate_idx in range(len(candidates)):

                if candidate_idx in indices_selectionnes:
                    continue

                essai = selection.copy()
                essai[position] = candidate_idx

                d_essai, eq_essai = evaluer_selection(essai)

                if (
                    np.isfinite(d_essai)
                    and d_essai < meilleur_d_local - 1e-10
                ):
                    meilleur_d_local = d_essai
                    meilleur_eq_local = eq_essai
                    meilleur_echange = essai

        # Arrêter si aucune substitution n'améliore le plan
        if meilleur_echange is None:
            break

        selection = meilleur_echange
        d_error = meilleur_d_local
        equilibre = meilleur_eq_local

    if d_error < meilleur_d_error:
        meilleur_d_error = d_error
        meilleur_equilibre = equilibre
        meilleur_plan = selection.copy()

# --------------------------------------------------
# 7. Afficher les 12 questions sélectionnées
# --------------------------------------------------

df_questions = candidates.iloc[meilleur_plan].copy()
df_questions = df_questions.reset_index(drop=True)

df_questions.insert(
    0, "Question", range(1, len(df_questions) + 1)
)

print("Nombre de questions :", len(df_questions))
print("D-error final :", meilleur_d_error)
print("Équilibre des utilités (%) :", meilleur_equilibre)

display(df_questions)


Nombre de questions : 12
D-error final : 0.5216948600244291
Équilibre des utilités (%) : 100.0


,Question,Prix_SM,Tailles_SM,Retours_SM,Prix_neuf,Tailles_neuf,Retours_neuf
0,1,100,1,1,150,2,2
1,2,100,1,2,300,2,1
2,3,100,1,2,300,1,1
3,4,100,1,2,150,2,1
4,5,150,1,1,300,1,2
5,6,150,1,2,300,1,1
6,7,100,1,1,300,2,2
7,8,150,1,1,300,2,2
8,9,150,1,1,100,2,2
9,10,150,1,2,100,2,1
